# 08 - 推理引擎 (AI Infra 视角)

本节从 **工程实现** 角度理解推理引擎：
- KV Cache 显存分析
- Prefill vs Decode 性能
- 采样策略实现
- 批处理优化

In [1]:
import torch
import torch.nn.functional as F

## 1. 推理引擎核心 (30秒版)

```
推理 = Prefill + Decode

Prefill (处理 prompt):
  - 一次处理所有 token
  - 填充 KV Cache
  - Compute Bound

Decode (生成):
  - 每次只处理 1 个 token
  - 查询 KV Cache
  - Memory Bound
```

## 2. KV Cache 显存分析 (重要!)

```
KV Cache = 2 × batch × seq × n_layers × n_kv_heads × head_dim × dtype

7B 模型 (32 layers, 32 heads, head_dim=128):
  batch=1, seq=4096:
  KV = 2 × 1 × 4096 × 32 × 32 × 128 × 2 = 2.1 GB

  batch=32, seq=4096:
  KV = 67 GB!  ← 超过模型本身!
```

In [2]:
def kv_cache_memory(batch, seq_len, n_layers, n_kv_heads, head_dim, dtype_bytes=2):
    """KV Cache 显存 (GB)"""
    mem = 2 * batch * seq_len * n_layers * n_kv_heads * head_dim * dtype_bytes
    return mem / 1e9

# 7B 模型配置
n_layers, n_kv_heads, head_dim = 32, 32, 128

print("7B 模型 KV Cache 显存:")
print("-" * 45)
for batch in [1, 8, 32, 64]:
    for seq_len in [2048, 4096, 8192]:
        mem = kv_cache_memory(batch, seq_len, n_layers, n_kv_heads, head_dim)
        print(f"  batch={batch:2d}, seq={seq_len:5d}: {mem:5.1f} GB")

7B 模型 KV Cache 显存:
---------------------------------------------
  batch= 1, seq= 2048:   1.1 GB
  batch= 1, seq= 4096:   2.1 GB
  batch= 1, seq= 8192:   4.3 GB
  batch= 8, seq= 2048:   8.6 GB
  batch= 8, seq= 4096:  17.2 GB
  batch= 8, seq= 8192:  34.4 GB
  batch=32, seq= 2048:  34.4 GB
  batch=32, seq= 4096:  68.7 GB
  batch=32, seq= 8192: 137.4 GB
  batch=64, seq= 2048:  68.7 GB
  batch=64, seq= 4096: 137.4 GB
  batch=64, seq= 8192: 274.9 GB


In [3]:
# GQA 对 KV Cache 的影响
print("GQA 对 KV Cache 的影响 (batch=32, seq=4096):")
print("-" * 50)

configs = [
    ("MHA (32 heads)", 32),
    ("GQA (8 heads)", 8),
    ("GQA (4 heads)", 4),
    ("MQA (1 head)", 1),
]

for name, n_kv_heads in configs:
    mem = kv_cache_memory(32, 4096, 32, n_kv_heads, 128)
    print(f"  {name:20s}: {mem:5.1f} GB")

GQA 对 KV Cache 的影响 (batch=32, seq=4096):
--------------------------------------------------
  MHA (32 heads)      :  68.7 GB
  GQA (8 heads)       :  17.2 GB
  GQA (4 heads)       :   8.6 GB
  MQA (1 head)        :   2.1 GB


## 3. Prefill vs Decode 性能

```
Prefill (Compute Bound):
  - 处理整个 prompt
  - 矩阵乘法: (B, T, D) × (D, D)
  - 充分利用 GPU 并行
  - 吞吐量高

Decode (Memory Bound):
  - 每次只处理 1 token
  - 矩阵乘法: (B, 1, D) × (D, D)
  - GPU 利用率低
  - 瓶颈在读取权重
```

In [5]:
def estimate_throughput(n_params, batch, gpu_memory_bw=2e12):
    """
    估算吞吐量
    
    gpu_memory_bw: H100 约 2TB/s
    """
    # 每个 token 需要读取整个模型
    bytes_per_token = n_params * 2  # BF16
    
    # Decode: Memory Bound
    # 每秒能读多少次模型 = 处理多少 token
    decode_tokens_per_sec = gpu_memory_bw / bytes_per_token * batch
    
    return decode_tokens_per_sec

# 7B 模型在 H100 上
n_params = 7e9

print("7B 模型 Decode 吞吐量估算 (H100):")
for batch in [1, 8, 32, 64]:
    tps = estimate_throughput(n_params, batch)
    print(f"  batch={batch:2d}: {tps:,.0f} tokens/sec")

7B 模型 Decode 吞吐量估算 (H100):
  batch= 1: 143 tokens/sec
  batch= 8: 1,143 tokens/sec
  batch=32: 4,571 tokens/sec
  batch=64: 9,143 tokens/sec


## 4. 采样策略

```
Temperature:  控制随机性
  - temp=0: 贪婪 (argmax)
  - temp=1: 正常采样
  - temp>1: 更随机

Top-k:  只从前 k 个采样
Top-p:  只从累积概率 p 内采样
```

In [6]:
@torch.inference_mode()
def sample_next_token(logits, temperature=1.0, top_k=None, top_p=None):
    """
    从 logits 采样下一个 token
    
    Args:
        logits: (batch, vocab_size)
        temperature: 温度
        top_k: 只考虑前 k 个
        top_p: 只考虑累积概率 p 内
    """
    # Temperature = 0 → 贪婪
    if temperature == 0.0:
        return logits.argmax(dim=-1, keepdim=True)
    
    # 应用 temperature
    logits = logits / temperature
    
    # Top-k 过滤
    if top_k is not None:
        k = min(top_k, logits.size(-1))
        vals, _ = torch.topk(logits, k, dim=-1)
        logits[logits < vals[:, -1:]] = float('-inf')
    
    # Top-p 过滤
    if top_p is not None:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cumsum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        mask = cumsum - F.softmax(sorted_logits, dim=-1) > top_p
        sorted_logits[mask] = float('-inf')
        logits = sorted_logits.gather(-1, sorted_idx.argsort(-1))
    
    # 采样
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

# 测试
logits = torch.randn(2, 50000)
print("采样结果:")
print(f"  Greedy:      {sample_next_token(logits, temperature=0).tolist()}")
print(f"  Normal:      {sample_next_token(logits, temperature=1).tolist()}")
print(f"  Top-k (10):  {sample_next_token(logits, top_k=10).tolist()}")

采样结果:
  Greedy:      [[44522], [36128]]
  Normal:      [[19890], [35543]]
  Top-k (10):  [[31116], [36128]]


## 5. KV Cache 实现

In [7]:
class KVCache:
    """
    KV Cache 管理
    
    shape: (n_layers, 2, batch, n_kv_heads, max_seq, head_dim)
                      ↑
                  0=K, 1=V
    """
    
    def __init__(self, n_layers, batch, n_kv_heads, max_seq, head_dim, device='cpu', dtype=torch.float16):
        self.cache = torch.zeros(
            n_layers, 2, batch, n_kv_heads, max_seq, head_dim,
            device=device, dtype=dtype
        )
        self.n_layers = n_layers
        self.pos = 0
    
    def get_pos(self):
        return self.pos
    
    def insert_kv(self, layer_idx, k, v):
        """插入新的 K, V，返回完整缓存"""
        T_new = k.size(2)
        t0, t1 = self.pos, self.pos + T_new
        
        self.cache[layer_idx, 0, :, :, t0:t1, :] = k
        self.cache[layer_idx, 1, :, :, t0:t1, :] = v
        
        if layer_idx == self.n_layers - 1:
            self.pos = t1
        
        return (
            self.cache[layer_idx, 0, :, :, :t1, :],
            self.cache[layer_idx, 1, :, :, :t1, :]
        )
    
    def reset(self):
        self.pos = 0

# 显存占用
cache = KVCache(n_layers=32, batch=1, n_kv_heads=32, max_seq=4096, head_dim=128)
print(f"KV Cache 显存: {cache.cache.numel() * 2 / 1e9:.2f} GB")

KV Cache 显存: 2.15 GB


## 6. 生成循环

In [ ]:
# nanochat 生成流程 (简化版)
generate_code = '''
def generate(model, prompt_ids, max_tokens, temperature=1.0, top_k=None):
    # 1. Prefill: 处理 prompt，填充 KV Cache
    kv_cache = KVCache(...)
    ids = torch.tensor([prompt_ids])
    logits = model.forward(ids, kv_cache=kv_cache)[:, -1, :]
    
    generated = list(prompt_ids)
    
    # 2. Decode: 逐个生成
    for _ in range(max_tokens):
        # 采样
        next_id = sample_next_token(logits, temperature, top_k)
        generated.append(next_id.item())
        
        # 检查终止
        if next_id.item() == eos_token_id:
            break
        
        # 只用新 token 计算 (利用 KV Cache)
        logits = model.forward(next_id, kv_cache=kv_cache)[:, -1, :]
    
    return generated
'''
print(generate_code)

## 7. 批处理优化

```
多样本生成:
  1. Prefill (batch=1): 处理 prompt
  2. 复制 KV Cache 给 N 个样本
  3. Decode (batch=N): 并行生成 N 个回复

优点:
  - Prefill 只做一次
  - Decode 时增加 batch 提高 GPU 利用率
```

In [8]:
# 多样本生成示例
print("多样本生成流程:")
print()
print("1. Prefill (batch=1):")
print("   prompt: 'What is 2+2?'")
print("   → KV Cache (1, heads, seq, dim)")
print()
print("2. 复制 KV Cache (num_samples=4):")
print("   → KV Cache (4, heads, seq, dim)")
print()
print("3. Decode (batch=4):")
print("   Sample 0: '2+2 = 4'")
print("   Sample 1: 'The answer is 4'")
print("   Sample 2: '4'")
print("   Sample 3: 'Two plus two equals four'")

多样本生成流程:

1. Prefill (batch=1):
   prompt: 'What is 2+2?'
   → KV Cache (1, heads, seq, dim)

2. 复制 KV Cache (num_samples=4):
   → KV Cache (4, heads, seq, dim)

3. Decode (batch=4):
   Sample 0: '2+2 = 4'
   Sample 1: 'The answer is 4'
   Sample 2: '4'
   Sample 3: 'Two plus two equals four'


## 8. 面试常见问题

### Q1: KV Cache 的作用?

**答**:
- 缓存历史 token 的 K, V，避免重复计算
- 无 Cache: O(n²) 计算
- 有 Cache: O(n) 计算
- 空间换时间

---

### Q2: Prefill 和 Decode 的区别?

**答**:
- **Prefill**: 处理 prompt，Compute Bound，吞吐量高
- **Decode**: 生成 token，Memory Bound，延迟敏感
- 优化方向不同：Prefill 优化计算，Decode 优化带宽

---

### Q3: 为什么 Decode 是 Memory Bound?

**答**:
- 每次只处理 1 个 token
- 矩阵乘法: (B, 1, D) × (D, D)
- 计算量小，但要读取整个模型
- 瓶颈在 GPU 显存带宽

---

### Q4: 如何提高 Decode 吞吐量?

**答**:
1. **增加 batch**: 提高 GPU 利用率
2. **GQA/MQA**: 减少 KV Cache，支持更大 batch
3. **量化**: INT8/INT4 减少显存带宽需求
4. **Continuous Batching**: 动态调度请求

---

### Q5: Temperature 如何影响生成?

**答**:
- temp=0: 贪婪，确定性，可能重复
- temp=1: 正常分布采样
- temp>1: 更随机，更有创意但可能离题
- temp<1: 更确定，更安全但可能无聊

---

### Q6: Top-k 和 Top-p 的区别?

**答**:
- **Top-k**: 固定只从前 k 个采样
- **Top-p**: 动态，累积概率达到 p 的最小集合
- Top-p 更灵活：高置信时只选 1 个，低置信时选多个

## 9. 总结速查表

| 主题 | 要点 |
|------|------|
| **KV Cache** | 2 × B × T × L × H × D × dtype |
| **Prefill** | Compute Bound，一次处理 prompt |
| **Decode** | Memory Bound，瓶颈在显存带宽 |
| **Temperature** | 0=贪婪，1=正常，>1=随机 |
| **优化** | 增加 batch, GQA, 量化 |

### 显存速算 (7B 模型)

```
模型参数:    14 GB (BF16)
KV Cache:    2 GB × batch × (seq/4096)

batch=1, seq=4096:  ~16 GB
batch=32, seq=4096: ~80 GB (需要 H100)
```